# Quantization Aware Training

In [ ]:
import os
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import DataLoader

from src.utils import load_data
from src.Quantization.utils.post_training_quantization import quantize_model
from src.Quantization.utils.preprocessing import load_kd_model
from src.utils import compute_loss_and_predictions, calculate_metrics
from src.utils import test_inference
from src.utils import model_size, measure_inference_performance

In [ ]:
# Specify random seed for repeatable results
_ = torch.manual_seed(191009)

### Helper Functions

In [ ]:
def load_quantized_model(model_path: str, device: torch.device = torch.device("cpu")) -> torch.jit.ScriptModule:
    """
    Loads a quantized TorchScript model from the specified file.

    Parameters:
        model_path (str): Path to the saved TorchScript model (.pt file).
        device (torch.device): The device on which to load the model. Defaults to CPU.
    
    Returns:
        torch.jit.ScriptModule: The loaded quantized model.
    """
    # Load the TorchScript model from disk, mapping it to the specified device.
    model = torch.jit.load(model_path, map_location=device)
    # Set the model to evaluation mode (important for inference)
    model.eval()
    return model


### Quantization Aware Training

In [ ]:

def train_qat_model(model: nn.Module, train_loader: DataLoader, device: torch.device, epochs: int = 10, lr: float = 1e-4) -> nn.Module:
    """
    Trains the QAT model using a standard training loop with cross entropy loss

    Parameters:
        model (nn.Module): The QAT-prepared model.
        train_loader (DataLoader): DataLoader for the training dataset.
        device (torch.device): Device to perform training on.
        epochs (int): Number of training epochs.
        lr (float): Learning rate.
    
    Returns:
        nn.Module: The trained QAT model.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()  # Ensure model is in training mode
        running_loss = 0.0

        # Create a progress bar for the current epoch
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
        
        for images, targets in progress_bar:
            images, targets = images.to(device), targets.to(device)
            
            optimizer.zero_grad()               # Reset gradients
            outputs = model(images)             # Forward pass
            loss = criterion(outputs, targets)  # Compute loss
            loss.backward()                     # Backward pass
            optimizer.step()                    # Update weights
            
            # Update running loss
            running_loss += loss.item() * images.size(0)
            
            # Update progress bar with current batch loss
            progress_bar.set_postfix(loss=loss.item())
        
        # Calculate and print epoch loss
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f}")
    
    return model

### Hyperparameters

In [ ]:
# Paths for the dataset and model weights
dataset = "SkinCancer"  # Update with your dataset path
batch_size = 32

dataloaders = load_data(dataset=dataset, batch_size=batch_size)
num_classes = len(dataloaders["train"].dataset.classes)

model_weights_path = 'models/SkinCancer/mobilenet_v2_best_model.pth'  # Update with your saved model weights

# Device configuration - quantization is often done on CPU.
device = torch.device("cpu")

# Load the pre-trained (fine-tuned) MobileNetV2 model
qat_model = load_kd_model("mobilenet_v2", None, num_classes)
qat_model.train()
qat_model.fuse_model(is_qat=True)
optimizer = torch.optim.SGD(qat_model.parameters(), lr = 0.0001)

# Use the custom qconfig in your FX quantization flow
# custom_qconfig = get_custom_qconfig()
# qat_model.qconfig = torch.ao.quantization.QConfigMapping().set_global(custom_qconfig)
qat_model.qconfig = torch.ao.quantization.get_default_qat_qconfig('x86')

torch.ao.quantization.prepare_qat(qat_model, inplace=True)
# print('Inverted Residual Block: After preparation for QAT, note fake-quantization modules \n',qat_model.features[1].conv)

model = train_qat_model(qat_model, dataloaders["train"], device, epochs=1, lr=1e-4)
